## Task 1 => API setup and basic prompt

Instead of Open AI using Chat Groq as its api key is available for free

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# load dotenv for ChatGroq API key
load_dotenv()

# Initialize Chat model 
llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

# Send a prompt to the model
response = llm.invoke('Explain RAG in simple words')

print(response.content)

## Task 2 => Wikipedia Retriever

In [ ]:
from langchain_classic.retrievers import WikipediaRetriever

# Initialize Wikipedia Retriever
retriever = WikipediaRetriever(top_k_results=3, lang='en')

query = 'Who is Mahatma Gandhi'

# Retrieve documents
result = retriever.invoke(query)

for i, doc in enumerate(result):
    print(f'Result {i+1}')
    print(doc.page_content)
    print()

## Task 3 => Vector stores Retriever

In [10]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Sample Doc
documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by language models.",
        metadata={"source": "LangChain overview", "topic": "LangChain"},
    ),
    Document(
        page_content="Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.",
        metadata={"source": "RAG overview", "topic": "RAG"},
    ),
    Document(
        page_content="A retriever searches a collection of documents and returns the ones most relevant to a query.",
        metadata={"source": "Retriever overview", "topic": "Retrievers"},
    ),
]

# Loading embedding model
embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

# Creating vector store 
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name='new_collection')

# Create retriever
vec_retriever = vector_store.as_retriever(search_kwargs={'k': 1})

query = 'what retriever do?'

similarity_results = vec_retriever.invoke(query)

for i, doc in enumerate(similarity_results):
    print(f'\nResult {i+1}')
    print(doc.page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3422.67it/s]



Result 1
A retriever searches a collection of documents and returns the ones most relevant to a query.


## Task 4 => MMR Retriever

In [11]:
from langchain_community.vectorstores import FAISS

# creating FAISS Vector Store using prev task documents and embeddings
faiss_vector_store = FAISS.from_documents(documents, embeddings)

# Create an MMR retriever
faiss_retriever = faiss_vector_store.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

query = 'what retriever do?'

mmr_results = faiss_retriever.invoke(query)

for i, doc in enumerate(mmr_results):
    print(f'\nResult {i+1}')
    print(doc.page_content)


Result 1
A retriever searches a collection of documents and returns the ones most relevant to a query.

Result 2
LangChain is a framework for building applications powered by language models.

Result 3
Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.


In [ ]:
## SIMILARITY SEARCH vs MMR

print("\nNORMAL SIMILARITY SEARCH:")
for i, doc in enumerate(similarity_results, start=1):
    print(f"{i}. {doc.metadata.get('source')}")
    print(doc.page_content)

print("\nMMR SEARCH:")
for i, doc in enumerate(mmr_results, start=1):
    print(f"{i}. {doc.metadata.get('source')}")
    print(doc.page_content)


NORMAL SIMILARITY SEARCH:
1. Retriever overview
A retriever searches a collection of documents and returns the ones most relevant to a query.

MMR SEARCH:
1. Retriever overview
A retriever searches a collection of documents and returns the ones most relevant to a query.
2. LangChain overview
LangChain is a framework for building applications powered by language models.
3. RAG overview
Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.


## MMR improves diversity by selecting documents that are relevant to the query but also different from documents already selected. This reduces repetitive information and provides a wider range of context for the LLM.
## The lambda_mult parameter controls the balance between relevance and diversity.

## Task 5 => Multi Query Retriever

In [15]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# Create FAISS vector store using Task 3 documents and embeddings
vector_store = FAISS.from_documents(documents, embeddings)

# Initialize Groq chat model
llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

# Create Multi-Query Retriever
multi_retriever = MultiQueryRetriever.from_llm(retriever=vector_store.as_retriever(), llm=llm)

query = 'what RAG do?'

# Retrieve documents using multiple query variations
multi_result = multi_retriever.invoke(query)

for i, doc in enumerate(multi_result):
    print(f'\nResult {i+1}')
    print(doc.page_content)

print("\nCOMPARISON")
print("Normal retrieval:", len(similarity_results), "documents")
print("Multi-query retrieval:", len(multi_result), "documents")


Result 1
Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.

Result 2
A retriever searches a collection of documents and returns the ones most relevant to a query.

Result 3
LangChain is a framework for building applications powered by language models.

COMPARISON
Normal retrieval: 1 documents
Multi-query retrieval: 3 documents


## Task 6 => Contextual Compression Retriever

In [19]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Create FAISS vector store using the same documents and embeddings
vector_store = FAISS.from_documents(documents, embeddings)

# Initialize Groq LLM
llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)

# Create the base retriever
base_retriever = vector_store.as_retriever()

# Create a compressor using the LLM
base_compressor = LLMChainExtractor.from_llm(llm)

# Combine retriever and compressor
compressor_retriever = ContextualCompressionRetriever(base_retriever=base_retriever, base_compressor=base_compressor)

query = "What does RAG do?"

# Before result
before_results = base_retriever.invoke(query)

print("BEFORE COMPRESSION")
for i, doc in enumerate(before_results):
    print(f"\nResult {i+1}:")
    print(doc.page_content)

# After result
after_results = compressor_retriever.invoke(query)

print("\nAFTER COMPRESSION")
for i, doc in enumerate(after_results):
    print(f"\nResult {i+1}:")
    print(doc.page_content)

BEFORE COMPRESSION

Result 1:
Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.

Result 2:
LangChain is a framework for building applications powered by language models.

Result 3:
A retriever searches a collection of documents and returns the ones most relevant to a query.

AFTER COMPRESSION

Result 1:
Retrieval augmented generation (RAG) retrieves relevant documents and uses them as context for a language model.

Result 2:
A retriever searches a collection of documents and returns the ones most relevant to a query.


## Task 7 => Laod YouTube Content

In [28]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize YouTube Transcript API
yt_api = YouTubeTranscriptApi()

# video ID from the YouTube URL
video_id = 'UabBYexBD4k'

# Fetch the transcript
transcript = yt_api.fetch(video_id)

# Combine transcript segments into one string
transcript_txt = " ".join(doc.text for doc in transcript)

## Text Splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# Split transcript into smaller chunks
transcript_chunks = splitter.split_text(transcript_txt)

for i, chunk in enumerate(transcript_chunks[:2]):
    print(f"\nChunk {i+1}:")
    print(chunk)



Chunk 1:
There's a fundamental truth about LLMs, large 
language models. They are frozen in time. They   know everything about our world up until 
their training cutoff date and absolutely   nothing about what happened 5 minutes ago. Nor 
do they know anything about your private data,   your internal wikis, your proprietary codebase. And 
if we do want an LLM to know any of that stuff,   well, we have to solve the problem of context 
injection. How do we get the right data into the   model at the right time? And there have been two 
very different ways to handle this. Now, the first   is really what we can think of as the engineering 
approach. It's RAG, retrieval augmented generation.   So here we've got an LLM and we've also got an 
input prompt from the user. Now ahead of time   we take some of the documents that we want to give 
to this LLM. So these are documents that could be   PDFs or code files or entire books and we chunk

Chunk 2:
to this LLM. So these are documents that cou

## Task 8 => Build Vector Store for YouTube Content

In [29]:
# Loading embedding model
embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

# creating vector store with previous task chunks and embeddings
vector_store = FAISS.from_texts(transcript_chunks, embeddings)

yt_retriever = vector_store.as_retriever(search_kwargs={'k':4})

# 5. Test the retriever
query = "is RAG still needed?"
yt_results = yt_retriever.invoke(query)

for i, doc in enumerate(yt_results, start=1):
    print(f"\nResult {i}:")
    print(doc.page_content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2972.72it/s]



Result 1:
long context wins on simplicity, RAG still has a   place. And I got another three reasons to support 
that. So, reason number one is the rereading text.   Now, long context creates a massive compute 
inefficiency. So, if we take a manual, let's say   this is like a a 500 page manual, and we've got to 
turn this into tokens. Well, that's something like   250k of tokens. And we need to do that every time 
we make a user query and we put this document in   the prompt. You're basically requiring the model 
to process that manual every time. Now, RAG also   has to process that manual, but it only pays that 
processing cost once at indexing time. Now, prompt   caching that can partially offset some of this for 
static data, but for dynamic data streams where   content changes frequently, you are stuck paying 
the full tax on every request. Reason number two   is the needle in the haystack problem. Now,

Result 2:
chunks discussing well security and requirements.   It retrieves sni

## Task 9 => Build YouTube RAG Chatbot

In [31]:
from langchain_core.prompts import PromptTemplate

# Augmentation
prompt = PromptTemplate(
    template="""You are a helpful assistant. Answer using the context.
If the context is insufficient, say "I don't know."

Context: {context}
Question: {question}""",
    input_variables=["context", "question"],
)

ques = 'How to choose best approach of LLM'

# Retrieval: Retrieve relevant chunks from YouTube transcript
context = yt_retriever.invoke(ques)

context = ' '.join(i.page_content for i in context)

# Insert context and question into the prompt
final_prompt = prompt.invoke({'question': ques, 'context': context})

## Generation
llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0.7)
response = llm.invoke(final_prompt)

print(response.content)

**Choosing the right way to feed data to an LLM boils down to three practical questions:**

| Question | What it tells you | Recommended approach |
|----------|-------------------|----------------------|
| **How big is the data you need to answer the query?** | • A *bounded* set (e.g., a single contract, a book, a handful of PDFs) that can fit—or almost fit—into the model’s context window.<br>• An *unbounded* or “infinite” set (enterprise knowledge base, terabytes of logs, a data lake). | **Bounded → Long‑context** (just dump the chunks into the prompt).<br>**Unbounded → Retrieval‑augmented generation (RAG)** with a vector DB. |
| **What kind of reasoning is required?** | • Complex, *global* reasoning that needs the model to see the whole document (e.g., cross‑clause analysis in a legal contract, summarizing an entire book).<br>• Local, *lookup‑style* reasoning (e.g., “What is the SLA for service X?”) where the answer lives in a small relevant fragment. | **Global reasoning → Long‑cont

## Task 10 => Testing and Evaluation

## Task 11 => Observation and Insights